# 📖 NB-01: GPT-2 Fine-tuning on Claude Thinking Dataset
**Goal:** Fine-tune GPT-2 to mimic Claude's response style using the 250-example dataset.

**Modality:** Text Generation | **Model:** GPT-2 / GPT-2 Medium

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer
from datasets import Dataset
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Format: "<USER>: {q}\n<ASSISTANT>: {a}<|endoftext|>"
texts = [f"<USER>: {d['user']}\n<ASSISTANT>: {d['response']}{tokenizer.eos_token}" for d in data]

def tokenize(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

ds = Dataset.from_dict({"text": texts}).map(tokenize, batched=True, remove_columns=["text"])
ds = ds.train_test_split(test_size=0.1)

args = TrainingArguments(
    output_dir="./gpt2-claude",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    logging_steps=20,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"])
trainer.train()
print("✅ GPT-2 fine-tuning complete!")


In [ ]:
# --- Inference ---
from transformers import pipeline
gen = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
prompt = "<USER>: What is machine learning?\n<ASSISTANT>:"
out = gen(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
print(out[0]["generated_text"])
